# PLAID to I.DOT notebook

In [1]:
# --- CELL 1: Setup (portable project root + outputs + user config) ---
# - Finds the project root by looking for expected files/folders
# - Sets the working directory to the project root so relative paths always work
# - Ensures outputs/ exists
# - Ensures config.json exists (creates a default once; never overwrites)
# - Loads helper functions from utils/my_functions.ipynb

from pathlib import Path
import os
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# --------------------
# Locate project root
# --------------------
HERE = Path.cwd().resolve()
ROOT_MARKERS = [
    "environment.yml",
    "meta/cmpd_info.csv",
    "utils/my_functions.ipynb",
    "notebooks",
]

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if all((p / m).exists() for m in ROOT_MARKERS):
            return p
    raise FileNotFoundError(
        "Project root not found.\n"
        "Expected to find these paths relative to the project root:\n"
        + "\n".join(f"  - {m}" for m in ROOT_MARKERS)
        + "\n\nTip: open the notebook from inside the project folder."
    )

PROJECT_DIR = find_project_root(HERE)
os.chdir(PROJECT_DIR)

# --------------------
# Outputs directory
# --------------------
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------
# User configuration (single edit point)
# --------------------
CONFIG_PATH = PROJECT_DIR / "config.json"
DEFAULT_CONFIG = {
    # Input selection (users only edit the filename)
    "layout_file": "Layout_1.csv",                 # must exist in inputs/plaid_layouts/
    "meta_file": "cmpd_info.csv",                  # must exist in meta/

    # Naming
    "user_name": "Taka",
    "protocol_name": "LALS_2026",

    # Plate settings
    "sourceplate_type": "S.100 Plate",             # S.60 Plate / S.100 Plate / S.200 Plate
    "target_plate_type": "MWP 384",                # MWP 96 / MWP 384

    # Assay constraints
    "working_volume_ul": 40,
    "max_dmso_pct": 0.1
}

if not CONFIG_PATH.exists():
    CONFIG_PATH.write_text(json.dumps(DEFAULT_CONFIG, indent=2) + "\n", encoding="utf-8")
    print(f"Created default config: {CONFIG_PATH}")
else:
    print(f"Using config: {CONFIG_PATH}")

# --------------------
# Load helper functions
# --------------------
%run utils/my_functions.ipynb

print("Project root :", PROJECT_DIR)
print("Outputs dir  :", OUTPUT_DIR)
print("Config file  :", CONFIG_PATH)

Using config: /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT/config.json
Project root : /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT
Outputs dir  : /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT/outputs
Config file  : /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT/config.json


In [3]:
# --- CELL 2: Configuration (load, validate, resolve paths, define outputs, load plate specs) ---
# - Loads user settings from config.json
# - Resolves input paths from simple filenames (layout_file, meta_file)
# - Validates files and parameter ranges
# - Defines output filenames that automatically include the layout name
# - Loads source-plate specifications based on SOURCEPLATE_TYPE (from utils/source_plate_specs.json)

import json
from pathlib import Path

# Load config
cfg = json.loads(Path(CONFIG_PATH).read_text(encoding="utf-8"))

# Required keys (single source of truth for user-editable settings)
required_keys = [
    "layout_file",
    "meta_file",
    "user_name",
    "protocol_name",
    "sourceplate_type",
    "target_plate_type",
    "working_volume_ul",
    "max_dmso_pct",
]
missing = [k for k in required_keys if k not in cfg]
if missing:
    raise KeyError(f"config.json is missing required keys: {missing}")

# Show available layout files to make selection easy
LAYOUT_DIR = PROJECT_DIR / "inputs" / "plaid_layouts"
available_layouts = sorted([p.name for p in LAYOUT_DIR.glob("*.csv")])
if not available_layouts:
    raise FileNotFoundError(f"No .csv layout files found in: {LAYOUT_DIR}")

# Resolve paths from filenames
layout_file = str(cfg["layout_file"])
meta_file = str(cfg["meta_file"])

PLAID_LAYOUT = (LAYOUT_DIR / layout_file).resolve()
META_PATH = (PROJECT_DIR / "meta" / meta_file).resolve()

# Validate paths exist
if not PLAID_LAYOUT.exists():
    raise FileNotFoundError(
        f"Layout file not found: {PLAID_LAYOUT}\n"
        f"Available layouts in {LAYOUT_DIR}:\n  - " + "\n  - ".join(available_layouts)
    )
if not META_PATH.exists():
    raise FileNotFoundError(f"Meta file not found: {META_PATH}")

# Parameters used downstream
USER_NAME = str(cfg["user_name"])
PROTOCOL_NAME = str(cfg["protocol_name"])
SOURCEPLATE_TYPE = str(cfg["sourceplate_type"])
TARGET_PLATE_TYPE = str(cfg["target_plate_type"])
WORKING_VOLUME_UL = float(cfg["working_volume_ul"])
MAX_DMSO_PCT = float(cfg["max_dmso_pct"])

# Sanity checks
if WORKING_VOLUME_UL <= 0:
    raise ValueError("working_volume_ul must be > 0")
if not (0 < MAX_DMSO_PCT <= 100):
    raise ValueError("max_dmso_pct must be in (0, 100]")

# Output naming: include layout stem for traceability
LAYOUT_TAG = PLAID_LAYOUT.stem
RUN_TAG = f"{PROTOCOL_NAME}__{LAYOUT_TAG}"

OUT_IDOT = OUTPUT_DIR / f"IDOT_{RUN_TAG}.csv"
OUT_LIQUIDS = OUTPUT_DIR / f"iDOT_liquids_{RUN_TAG}.csv"

# ---- Load source plate specifications ----
SPECS_PATH = PROJECT_DIR / "utils" / "source_plate_specs.json"
if not SPECS_PATH.exists():
    raise FileNotFoundError(f"Missing source plate specs file: {SPECS_PATH}")

specs = json.loads(SPECS_PATH.read_text(encoding="utf-8"))
if SOURCEPLATE_TYPE not in specs:
    raise ValueError(
        f"sourceplate_type '{SOURCEPLATE_TYPE}' not found in {SPECS_PATH}.\n"
        f"Available: {sorted(specs.keys())}"
    )

SOURCE_SPECS = specs[SOURCEPLATE_TYPE]

# Backward-compatible key access (supports both old and enriched spec formats)
SOURCE_PLATE_WELLS = int(SOURCE_SPECS["wells"])

# Prefer enriched keys; fall back to older names if present
SOURCE_EFFECTIVE_RESERVOIR_UL = float(
    SOURCE_SPECS.get("effective_reservoir_uL", SOURCE_SPECS.get("suggested_fill_uL", 80))
)

SOURCE_DEAD_UL_LT = float(
    SOURCE_SPECS.get("dead_volume_uL_aq_lt", SOURCE_SPECS.get("dead_volume_uL_min", 1))
)

SOURCE_DISPENSE_MIN_NL_AQ = float(SOURCE_SPECS.get("dispense_min_nL_aq", 0))
SOURCE_DISPENSE_MAX_NL_AQ = float(SOURCE_SPECS.get("dispense_max_nL_aq", 80000))

# Protocol header max volume (fallback keeps current behavior)
SOURCE_MAX_VOLUME_L_FOR_PROTOCOL = float(SOURCE_SPECS.get("max_volume_L_for_protocol", 8.0e-5))

print("Available layouts:", ", ".join(available_layouts))
print("Using layout     :", PLAID_LAYOUT)
print("Using meta       :", META_PATH)
print("Protocol         :", PROTOCOL_NAME)
print("Plates           :", SOURCEPLATE_TYPE, "->", TARGET_PLATE_TYPE)
print("Working volume   :", WORKING_VOLUME_UL, "uL")
print("Max DMSO         :", MAX_DMSO_PCT, "%")
print("Output protocol  :", OUT_IDOT)
print("Output liquids   :", OUT_LIQUIDS)
print("Source specs     :", SOURCE_SPECS)

Available layouts: Layout_1.csv, Layout_2.csv
Using layout     : /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT/inputs/plaid_layouts/Layout_1.csv
Using meta       : /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT/meta/cmpd_info.csv
Protocol         : LALS_2026
Plates           : S.100 Plate -> MWP 384
Working volume   : 40.0 uL
Max DMSO         : 0.1 %
Output protocol  : /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT/outputs/IDOT_LALS_2026__Layout_1.csv
Output liquids   : /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT/outputs/iDOT_liquids_LALS_2026__Layout_1.csv
Source specs     : {'wells': 96, 'orifice_um': 100, 'dispense_min_nL_aq': 8, 'dispense_max_nL_aq': 80000, 'dead_volume_uL_aq_lt': 1, 'effective_reservoir_uL': 80, 'max_volume_L_for_protocol': 8e-05, 'notes': 'Planning defaults. effective_reservoir_uL and max_volume_L_for_protocol are used for run planning/output header.'}


In [4]:
# --- CELL 3: Load layout + normalize core columns ---
# - Reads the selected PLAID layout CSV
# - Validates required columns
# - Normalizes identifiers (cmpdname, well, plateID)
# - Parses CONCuM robustly and preserves DMSO controls (missing CONCuM -> 0 µM)
# - Drops rows that cannot be dispensed

df = pd.read_csv(PLAID_LAYOUT)

# ---- Validate schema (fail fast) ----
required_cols = {"plateID", "well", "cmpdname", "CONCuM"}
missing_cols = sorted(required_cols - set(df.columns))
if missing_cols:
    raise ValueError(f"Layout file is missing required columns: {missing_cols}\nFile: {PLAID_LAYOUT}")

# ---- Normalize identifiers ----
df["cmpdname"] = (
    df["cmpdname"]
      .astype(str)
      .str.strip()
      .replace({"Dimethyl sulfoxide": "DMSO"})  # canonical solvent-control name
)
df["well"] = df["well"].astype(str).str.strip()
df["plateID"] = df["plateID"].astype(str).str.strip()

# ---- Parse CONCuM to numeric (robust to quotes/blanks/"nan") ----
df["CONCuM"] = (
    df["CONCuM"]
      .astype(str)
      .str.replace('"', "", regex=False)
      .str.strip()
      .replace({"nan": np.nan, "": np.nan, "None": np.nan})
)
df["CONCuM"] = pd.to_numeric(df["CONCuM"], errors="coerce")

# ---- Preserve DMSO controls (missing concentration -> 0 µM) ----
is_dmso_control = (
    df["cmpdname"].str.lower().eq("dmso")
    | df.get("treatment_type", pd.Series("", index=df.index)).astype(str).str.upper().str.contains("DMSO", na=False)
)
df.loc[is_dmso_control, "CONCuM"] = df.loc[is_dmso_control, "CONCuM"].fillna(0)

# ---- Drop rows that still cannot be dispensed ----
df = df.dropna(subset=["well", "CONCuM"]).copy()

print(f"Loaded rows: {len(df)}")
print(f"DMSO control wells retained: {int(is_dmso_control.sum())}")
display(df[["plateID", "well", "cmpdname", "CONCuM"]].head(10))

Loaded rows: 60
DMSO control wells retained: 24


,plateID,well,cmpdname,CONCuM
0,plate_1,B02,DMSO,0.10
1,plate_1,B03,Fenbendazole,1.00
2,plate_1,B04,Sorbitol,0.10
3,plate_1,B05,Etoposide,0.10
4,plate_1,B06,DMSO,0.10
5,plate_1,B07,Etoposide,0.01
6,plate_1,B08,DMSO,0.10
7,plate_1,B09,Fenbendazole,1.00
8,plate_1,B10,Berberine chloride,0.10
9,plate_1,B11,DMSO,0.10


In [5]:
# --- CELL 4: Merge compound metadata (stock + solvent) ---
# - Loads meta/cmpd_info.csv
# - Normalizes compound names
# - Merges solvent and highest_stock_mM onto the layout by cmpdname
# - Fails fast if any compounds are missing metadata

cmpd_info = pd.read_csv(META_PATH)

# Validate meta schema
meta_required = {"cmpdname", "highest_stock_mM", "solvent"}
meta_missing = sorted(meta_required - set(cmpd_info.columns))
if meta_missing:
    raise ValueError(f"Meta file is missing required columns: {meta_missing}\nFile: {META_PATH}")

# Normalize naming for robust joining
cmpd_info["cmpdname"] = cmpd_info["cmpdname"].astype(str).str.strip()

# Merge metadata into the layout
df = df.merge(cmpd_info, on="cmpdname", how="left", indicator=True)

# Report merge status
merge_counts = df["_merge"].value_counts()
print(merge_counts)

missing_cmpds = df.loc[df["_merge"] != "both", "cmpdname"].drop_duplicates().tolist()
if missing_cmpds:
    raise ValueError(
        "These compounds are missing from meta/cmpd_info.csv:\n  - "
        + "\n  - ".join(missing_cmpds)
    )

df = df.drop(columns=["_merge"]).copy()

# Enforce numeric type for stock concentrations
df["highest_stock_mM"] = pd.to_numeric(df["highest_stock_mM"], errors="raise")

display(df[["cmpdname", "CONCuM", "highest_stock_mM", "solvent"]].head(10))

_merge
both          60
left_only      0
right_only     0
Name: count, dtype: int64


,cmpdname,CONCuM,highest_stock_mM,solvent
0,DMSO,0.10,0,DMSO
1,Fenbendazole,1.00,10,DMSO
2,Sorbitol,0.10,10,DMSO
3,Etoposide,0.10,10,DMSO
4,DMSO,0.10,0,DMSO
5,Etoposide,0.01,10,DMSO
6,DMSO,0.10,0,DMSO
7,Fenbendazole,1.00,10,DMSO
8,Berberine chloride,0.10,10,DMSO
9,DMSO,0.10,0,DMSO


In [6]:
# --- CELL 5: Choose a feasible stock concentration per well ---
# - For each target concentration (CONCuM), selects a stock concentration (mM) that is dispensable
# - Respects the hard DMSO limit (MAX_DMSO_PCT) given the working volume (WORKING_VOLUME_UL)
# - Writes the chosen stock into df["stock_conc_mM"]
# - Uses a small wrapper to make SOURCEPLATE_TYPE explicit (no hidden global dependencies)

from utils.idot_helpers import stockfinder_safe

df["stock_conc_mM"] = df.apply(
    lambda r: stockfinder_safe(
        stockfinder,
        concUM=r["CONCuM"],
        highest_stock_mM=r["highest_stock_mM"],
        V2_ul=WORKING_VOLUME_UL,
        dmso_percmax=MAX_DMSO_PCT,
        sourceplate_type=SOURCEPLATE_TYPE,
    ),
    axis=1
)

# Fail fast if any wells could not be assigned a valid stock concentration
if df["stock_conc_mM"].isna().any():
    display(df.loc[df["stock_conc_mM"].isna(), ["plateID", "well", "cmpdname", "CONCuM", "highest_stock_mM"]])
    raise ValueError("Stock selection failed for some rows. Check MAX_DMSO_PCT, stocks, and requested CONCuM.")

# Quick summary (useful for spotting unexpected dilution patterns)
stock_summary = (
    df[["cmpdname", "stock_conc_mM"]]
    .value_counts()
    .to_frame("count")
    .reset_index()
)
display(stock_summary.head(30))

,cmpdname,stock_conc_mM,count
0,DMSO,0.00,24
1,Fenbendazole,1.00,3
2,Sorbitol,0.10,3
3,Etoposide,0.10,3
4,Etoposide,0.01,3
5,Berberine chloride,0.10,3
6,Etoposide,1.00,3
7,Fenbendazole,0.01,3
8,Sorbitol,0.01,3
9,Sorbitol,1.00,3


In [7]:
# --- CELL 6: Build dispense-ready columns ---
# - Standardizes target identifiers used in the protocol output
# - Creates a stable liquid identifier: [compound][stock_mM]
# - Computes dispense volumes (µL) for each well
# - Enforces the per-well DMSO ceiling (since stocks are in DMSO)

# Target identifiers used by the protocol
df["Target Plate"] = df["plateID"].astype(str)
df["Target Well"] = df["well"].astype(str).map(removeleadingzero)  # e.g., B03 -> B3

# Liquid identifier (must match downstream parsing: [Compound][Stock])
df["Liquid Name"] = "[" + df["cmpdname"].astype(str) + "][" + df["stock_conc_mM"].astype(str) + "]"

# Dispense volume (µL)
df["Volume [uL]"] = df.apply(
    lambda r: uLfromstock(r["CONCuM"], r["stock_conc_mM"], WORKING_VOLUME_UL),
    axis=1
)

# Volume calculation must succeed for all rows
if df["Volume [uL]"].isna().any():
    display(df.loc[df["Volume [uL]"].isna(), ["plateID", "well", "cmpdname", "CONCuM", "stock_conc_mM"]])
    raise ValueError("Volume calculation failed for some rows. Check CONCuM/stock_conc_mM and uLfromstock().")

# DMSO cap (µL) for a single well
max_dmso_ul = (MAX_DMSO_PCT / 100.0) * WORKING_VOLUME_UL
too_high = df.loc[df["Volume [uL]"] > max_dmso_ul]
if len(too_high) > 0:
    display(too_high[["Target Plate", "Target Well", "cmpdname", "CONCuM", "stock_conc_mM", "Volume [uL]"]])
    raise ValueError(f"DMSO limit exceeded: Volume [uL] > {max_dmso_ul:.6f} uL (>{MAX_DMSO_PCT}% of {WORKING_VOLUME_UL} uL).")

print(
    "Volume [uL] (min/median/max):",
    float(df["Volume [uL]"].min()),
    float(df["Volume [uL]"].median()),
    float(df["Volume [uL]"].max())
)
display(df[["Target Plate", "Target Well", "cmpdname", "CONCuM", "stock_conc_mM", "Liquid Name", "Volume [uL]"]].head(15))

Volume [uL] (min/median/max): 0.0 0.04 0.04


,Target Plate,Target Well,cmpdname,CONCuM,stock_conc_mM,Liquid Name,Volume [uL]
0,plate_1,B2,DMSO,0.10,0.00,[DMSO][0.0],0.00
1,plate_1,B3,Fenbendazole,1.00,1.00,[Fenbendazole][1.0],0.04
2,plate_1,B4,Sorbitol,0.10,0.10,[Sorbitol][0.1],0.04
3,plate_1,B5,Etoposide,0.10,0.10,[Etoposide][0.1],0.04
4,plate_1,B6,DMSO,0.10,0.00,[DMSO][0.0],0.00
5,plate_1,B7,Etoposide,0.01,0.01,[Etoposide][0.01],0.04
6,plate_1,B8,DMSO,0.10,0.00,[DMSO][0.0],0.00
7,plate_1,B9,Fenbendazole,1.00,1.00,[Fenbendazole][1.0],0.04
8,plate_1,B10,Berberine chloride,0.10,0.10,[Berberine chloride][0.1],0.04
9,plate_1,B11,DMSO,0.10,0.00,[DMSO][0.0],0.00


In [8]:
# --- CELL 6.5: DMSO normalization (constant across wells, within hard cap) ---
# - Computes DMSO already delivered by compound dispenses
# - Chooses a single final DMSO amount for the run (max already required by any well)
# - Computes DMSO top-up volumes to bring every well to the same final DMSO
# - Forces solvent-control wells (DMSO) to receive only top-up, matching the same final DMSO
# - Verifies constant DMSO across wells and compliance with MAX_DMSO_PCT

# Hard cap in µL (e.g., 0.1% of 40 µL = 0.04 µL)
max_dmso_ul = (MAX_DMSO_PCT / 100.0) * WORKING_VOLUME_UL

# DMSO contributed by compound dispenses (µL); non-DMSO solvents contribute 0
df["dmso_from_compound_uL"] = np.where(
    df["solvent"].astype(str).str.upper().eq("DMSO"),
    df["Volume [uL]"].astype(float),
    0.0
)

# Target final DMSO is the maximum already required by any well
target_dmso_ul = float(df["dmso_from_compound_uL"].max())

if target_dmso_ul > max_dmso_ul + 1e-12:
    raise ValueError(
        f"DMSO normalization impossible under the current cap:\n"
        f"  required max DMSO = {target_dmso_ul:.6f} uL\n"
        f"  allowed cap       = {max_dmso_ul:.6f} uL ({MAX_DMSO_PCT}% of {WORKING_VOLUME_UL} uL)"
    )

# DMSO needed to bring each well up to the target final DMSO
df["dmso_topup_uL"] = (target_dmso_ul - df["dmso_from_compound_uL"]).clip(lower=0.0)

# Solvent-control wells: no compound contribution; all DMSO comes from top-up
is_dmso_control = (
    df["cmpdname"].astype(str).str.strip().str.lower().eq("dmso")
    | df.get("treatment_type", pd.Series("", index=df.index)).astype(str).str.upper().str.contains("DMSO", na=False)
)
df.loc[is_dmso_control, "dmso_from_compound_uL"] = 0.0
df.loc[is_dmso_control, "dmso_topup_uL"] = target_dmso_ul

# Final DMSO per well (must be constant and within cap)
df["dmso_total_uL"] = df["dmso_from_compound_uL"] + df["dmso_topup_uL"]

if not np.allclose(df["dmso_total_uL"].values, target_dmso_ul, atol=1e-9):
    raise ValueError("DMSO normalization failed: final DMSO is not identical across all wells.")

if df["dmso_total_uL"].max() > max_dmso_ul + 1e-12:
    raise ValueError("DMSO normalization failed: exceeded MAX_DMSO_PCT cap.")

print(
    f"Target final DMSO per well: {target_dmso_ul:.6f} uL "
    f"({(target_dmso_ul / WORKING_VOLUME_UL) * 100:.4f}% of {WORKING_VOLUME_UL} uL)"
)

display(df[["Target Plate", "Target Well", "cmpdname", "Volume [uL]", "dmso_topup_uL", "dmso_total_uL"]].head(15))

Target final DMSO per well: 0.040000 uL (0.1000% of 40.0 uL)


,Target Plate,Target Well,cmpdname,Volume [uL],dmso_topup_uL,dmso_total_uL
0,plate_1,B2,DMSO,0.00,0.04,0.04
1,plate_1,B3,Fenbendazole,0.04,0.00,0.04
2,plate_1,B4,Sorbitol,0.04,0.00,0.04
3,plate_1,B5,Etoposide,0.04,0.00,0.04
4,plate_1,B6,DMSO,0.00,0.04,0.04
5,plate_1,B7,Etoposide,0.04,0.00,0.04
6,plate_1,B8,DMSO,0.00,0.04,0.04
7,plate_1,B9,Fenbendazole,0.04,0.00,0.04
8,plate_1,B10,Berberine chloride,0.04,0.00,0.04
9,plate_1,B11,DMSO,0.00,0.04,0.04


In [9]:
# --- CELL 7: Build protocol tables + write Assay Studio CSV ---
# - Combines compound dispenses and DMSO top-ups into one dispense table
# - Removes zero-volume rows (these are non-actions and can break strict imports)
# - Assigns a source well per unique liquid on a single source plate
# - Orders liquids by (compound, stock concentration) so dilutions cluster (0.01 → 0.1 → 1.0)
# - Orders dispense rows consistently for easy source-plate preparation
# - Uses the selected SOURCEPLATE_TYPE specs (loaded in Cell 2) for protocol header values
# - Writes:
#     IDOT_<protocol>__<layout>.csv  (Assay Studio import)
#     iDOT_liquids_<protocol>__<layout>.csv  (source plate map)

import datetime

# ---- Header metadata ----
x = datetime.datetime.now()
SOFTWARE = "1.7.2021.1019"
DATE = x.strftime("%x")
TIME = x.strftime("%X")

# ---- Assay Studio block settings ----
# Use source-plate spec for the protocol header (fallback keeps current behavior)
MAX_VOLUME_L = float(SOURCE_SPECS.get("max_volume_L_for_protocol", 8.0E-5))
WASTE_POS = "Waste Tube"

DISPENSE_TO_WASTE = True
DISPENSE_TO_WASTE_CYCLES = 2
DISPENSE_TO_WASTE_VOLUME_L = 5e-8
USE_DEIONISATION = True
OPTIMIZATION_LEVEL = "ReorderAndParallel"
WASTE_ERROR_HANDLING_LEVEL = "Ask"
SAVE_LIQUIDS = "Ask"

# ---- Dispense rows: compounds ----
compound_rows = df[["Target Plate", "Target Well", "Liquid Name", "Volume [uL]"]].copy()
compound_rows["Volume [uL]"] = compound_rows["Volume [uL]"].astype(float)

# ---- Dispense rows: DMSO top-up ----
DMSO_LIQUID_NAME = "[DMSO][0.0]"
dmso_rows = df.loc[df["dmso_topup_uL"] > 0, ["Target Plate", "Target Well", "dmso_topup_uL"]].copy()
dmso_rows = dmso_rows.rename(columns={"dmso_topup_uL": "Volume [uL]"})
dmso_rows["Liquid Name"] = DMSO_LIQUID_NAME
dmso_rows["Volume [uL]"] = dmso_rows["Volume [uL]"].astype(float)

# ---- Combine and remove non-actions ----
all_rows = pd.concat([compound_rows, dmso_rows], ignore_index=True)
all_rows = all_rows.loc[all_rows["Volume [uL]"] != 0].copy()

# ---- Source plate well IDs ----
def wells_96():
    rows = list("ABCDEFGH")
    cols = list(range(1, 13))
    return [f"{r}{c}" for c in cols for r in rows]  # column-major

# ---- Build liquid table and sort by compound + numeric stock concentration ----
liquid_table = all_rows[["Liquid Name"]].drop_duplicates().copy()

# Parse: [Compound][Stock]
liquid_table[["compound", "stock_str"]] = liquid_table["Liquid Name"].str.extract(r"^\[(.*?)\]\[(.*?)\]$")
bad = liquid_table.loc[liquid_table[["compound", "stock_str"]].isna().any(axis=1), "Liquid Name"]
if len(bad) > 0:
    raise ValueError(f"Liquid Name not in expected format [Compound][Stock]:\n{bad.to_list()}")

liquid_table["stock_mM"] = pd.to_numeric(liquid_table["stock_str"], errors="raise")
liquid_table["is_dmso"] = (liquid_table["compound"].str.upper() == "DMSO").astype(int)

liquid_table = liquid_table.sort_values(
    ["is_dmso", "compound", "stock_mM", "Liquid Name"],
    kind="mergesort"
).reset_index(drop=True)

# ---- Assign source wells ----
liquid_table["Source Plate"] = f"SRC_{PROTOCOL_NAME}"
avail = wells_96()
if len(liquid_table) > len(avail):
    raise ValueError(
        f"Too many unique liquids ({len(liquid_table)}) for one source plate ({len(avail)} wells)."
    )

liquid_table["Source Well"] = avail[:len(liquid_table)]

# Export map (clean columns only)
liquid_table_export = liquid_table[["Liquid Name", "Source Plate", "Source Well"]].copy()

# Attach source mapping and sort keys to dispense rows
all_rows = all_rows.merge(liquid_table_export, on="Liquid Name", how="left")
all_rows = all_rows.merge(liquid_table[["Liquid Name", "compound", "stock_mM", "is_dmso"]], on="Liquid Name", how="left")

# Sort dispenses to cluster liquids and keep targets ordered
all_rows = all_rows.sort_values(
    by=["is_dmso", "compound", "stock_mM", "Source Well", "Target Plate", "Target Well"],
    kind="mergesort"
).reset_index(drop=True)

# ---- Assemble Assay Studio CSV blocks ----
blocks = []
sourceplates = liquid_table_export["Source Plate"].unique().tolist()
targetplates = all_rows["Target Plate"].unique().tolist()

for sp in sourceplates:
    for tp in targetplates:
        dfx = all_rows.loc[(all_rows["Source Plate"] == sp) & (all_rows["Target Plate"] == tp)].copy()
        if dfx.empty:
            continue

        body = dfx[["Source Well", "Target Well", "Volume [uL]", "Liquid Name"]].copy()
        body["Volume [uL]"] = body["Volume [uL]"].map(lambda v: f"{float(v):05.2f}")

        body = body.reindex(columns=[*body.columns.tolist(), "", "", "", ""], fill_value="")
        body = pd.concat([body.columns.to_frame().T, body], ignore_index=True)
        body.columns = range(len(body.columns))

        subheader = pd.DataFrame([
            [SOURCEPLATE_TYPE, sp, "", MAX_VOLUME_L, TARGET_PLATE_TYPE, tp, "", WASTE_POS],
            [
                f"DispenseToWaste={DISPENSE_TO_WASTE}",
                f"DispenseToWasteCycles={DISPENSE_TO_WASTE_CYCLES}",
                f"DispenseToWasteVolume={DISPENSE_TO_WASTE_VOLUME_L}",
                f"UseDeionisation={USE_DEIONISATION}",
                f"OptimizationLevel={OPTIMIZATION_LEVEL}",
                f"WasteErrorHandlingLevel={WASTE_ERROR_HANDLING_LEVEL}",
                f"SaveLiquids={SAVE_LIQUIDS}",
                ""
            ],
        ])

        blocks.append(pd.concat([subheader, body], ignore_index=True))

file_header = pd.DataFrame([[PROTOCOL_NAME, SOFTWARE, USER_NAME, DATE, TIME, "", "", ""]])
fullprotocol = pd.concat([file_header, *blocks], ignore_index=True)

# Write outputs (paths defined in Cell 2)
fullprotocol.to_csv(OUT_IDOT, header=False, index=False, encoding="utf-8-sig")
print("Wrote Assay Studio protocol:", OUT_IDOT)

liquid_table_export.to_csv(OUT_LIQUIDS, index=False)
print("Wrote liquids map:", OUT_LIQUIDS)

display(fullprotocol.head(12))
display(liquid_table_export)

Wrote Assay Studio protocol: /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT/outputs/IDOT_LALS_2026__Layout_1.csv
Wrote liquids map: /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT/outputs/iDOT_liquids_LALS_2026__Layout_1.csv


,0,1,2,3,4,5,6,7
0,LALS_2026,1.7.2021.1019,Taka,03/04/26,16:31:43,,,
1,S.100 Plate,SRC_LALS_2026,,0.00008,MWP 384,plate_1,,Waste Tube
2,DispenseToWaste=True,DispenseToWasteCycles=2,DispenseToWasteVolume=5e-08,UseDeionisation=True,OptimizationLevel=ReorderAndParallel,WasteErrorHandlingLevel=Ask,SaveLiquids=Ask,
3,Source Well,Target Well,Volume [uL],Liquid Name,,,,
4,A1,D8,00.04,[Berberine chloride][0.01],,,,
5,A1,E6,00.04,[Berberine chloride][0.01],,,,
6,A1,F10,00.04,[Berberine chloride][0.01],,,,
7,B1,B10,00.04,[Berberine chloride][0.1],,,,
8,B1,D11,00.04,[Berberine chloride][0.1],,,,
9,B1,G3,00.04,[Berberine chloride][0.1],,,,


,Liquid Name,Source Plate,Source Well
0,[Berberine chloride][0.01],SRC_LALS_2026,A1
1,[Berberine chloride][0.1],SRC_LALS_2026,B1
2,[Berberine chloride][1.0],SRC_LALS_2026,C1
3,[Etoposide][0.01],SRC_LALS_2026,D1
4,[Etoposide][0.1],SRC_LALS_2026,E1
5,[Etoposide][1.0],SRC_LALS_2026,F1
6,[Fenbendazole][0.01],SRC_LALS_2026,G1
7,[Fenbendazole][0.1],SRC_LALS_2026,H1
8,[Fenbendazole][1.0],SRC_LALS_2026,A2
9,[Sorbitol][0.01],SRC_LALS_2026,B2


In [10]:
# --- CELL 8: Final validation (export format + DMSO constraints) ---
# - Confirms the generated protocol CSV exists and is non-empty
# - Verifies key structural markers required for a successful Assay Studio import
# - Verifies DMSO constraints:
#     (1) identical final DMSO per well across the plate
#     (2) final DMSO does not exceed MAX_DMSO_PCT of the working volume

import numpy as np
import pandas as pd

# Use the output path defined in Cell 2
out_idot = OUT_IDOT

# 1) File exists and is not empty
out_idot = str(out_idot)  # Path -> str for some OS calls
import os
assert os.path.exists(out_idot), f"Missing file: {out_idot}"
assert os.path.getsize(out_idot) > 0, f"Empty file: {out_idot}"
print("OK: file exists:", out_idot)

# 2) Structural checks on the exported CSV (headerless; inspect first lines)
p = pd.read_csv(out_idot, header=None, nrows=30)

# Header row: protocol metadata
assert str(p.iloc[0, 0]).strip() == PROTOCOL_NAME, "Header mismatch: protocol name"
assert str(p.iloc[0, 1]).strip() == "1.7.2021.1019", "Header mismatch: software version"
assert str(p.iloc[0, 2]).strip() == USER_NAME, "Header mismatch: user name"

# Expected fixed column count (4 table columns + 4 empty padding columns)
assert p.shape[1] == 8, f"Expected 8 columns, found {p.shape[1]}"

# Transfer-table header row must be present near the top
header_row_idx = None
for i in range(len(p)):
    row = p.iloc[i].astype(str).tolist()
    if ("Source Well" in row) and ("Target Well" in row) and ("Volume [uL]" in row) and ("Liquid Name" in row):
        header_row_idx = i
        break
assert header_row_idx is not None, "Did not find the transfer table header row"

print("OK: file structure looks correct. Transfer table header row at line:", header_row_idx)
display(p.head(12))

# 3) DMSO normalization checks (hard cap + constant across wells)
max_dmso_ul = (MAX_DMSO_PCT / 100.0) * WORKING_VOLUME_UL

assert "dmso_total_uL" in df.columns, "Missing df['dmso_total_uL'] (run Cell 6.5)"
target_dmso_ul = float(df["dmso_total_uL"].iloc[0])

if not np.allclose(df["dmso_total_uL"].values, target_dmso_ul, atol=1e-9):
    raise ValueError("DMSO normalization failed: final DMSO is not identical across all wells.")

if df["dmso_total_uL"].max() > max_dmso_ul + 1e-12:
    raise ValueError(
        f"DMSO exceeds cap: max {df['dmso_total_uL'].max():.6f} uL > cap {max_dmso_ul:.6f} uL"
    )

print(
    f"OK: constant DMSO per well = {target_dmso_ul:.6f} uL "
    f"({(target_dmso_ul / WORKING_VOLUME_UL) * 100:.4f}% of {WORKING_VOLUME_UL} uL) "
    f"<= {MAX_DMSO_PCT}% cap ✅"
)

OK: file exists: /Users/takar834/Documents/UU/TIMED/Analysis/PLAID_iDOT/outputs/IDOT_LALS_2026__Layout_1.csv
OK: file structure looks correct. Transfer table header row at line: 3


,0,1,2,3,4,5,6,7
0,LALS_2026,1.7.2021.1019,Taka,03/04/26,16:31:43,NaN,NaN,NaN
1,S.100 Plate,SRC_LALS_2026,NaN,8e-05,MWP 384,plate_1,NaN,Waste Tube
2,DispenseToWaste=True,DispenseToWasteCycles=2,DispenseToWasteVolume=5e-08,UseDeionisation=True,OptimizationLevel=ReorderAndParallel,WasteErrorHandlingLevel=Ask,SaveLiquids=Ask,NaN
3,Source Well,Target Well,Volume [uL],Liquid Name,NaN,NaN,NaN,NaN
4,A1,D8,00.04,[Berberine chloride][0.01],NaN,NaN,NaN,NaN
5,A1,E6,00.04,[Berberine chloride][0.01],NaN,NaN,NaN,NaN
6,A1,F10,00.04,[Berberine chloride][0.01],NaN,NaN,NaN,NaN
7,B1,B10,00.04,[Berberine chloride][0.1],NaN,NaN,NaN,NaN
8,B1,D11,00.04,[Berberine chloride][0.1],NaN,NaN,NaN,NaN
9,B1,G3,00.04,[Berberine chloride][0.1],NaN,NaN,NaN,NaN


OK: constant DMSO per well = 0.040000 uL (0.1000% of 40.0 uL) <= 0.1% cap ✅
